# 01c · A first look at Meteosat-9 IODC over India (Phase 1, task 1.3)

Exploration only. Uses the 3 slots of 15 Jan 2026 ingested on the laptop
(`uv run stratum meteosat-day --day 2026-01-15 --max-slots 3`) and the static grid.
Checks: viewing zenith angle (VZA) grows towards the east (paper §4.4; Mumbai ~38°,
Delhi ~48°, Kolkata ~55°), and the 60° observability limit cuts off the far north-east.
The coastline/geolocation check is `reports/phase1/meteosat/ir108_coast_2026-01-15T0000.png`.

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from stratum.io import meteosat
from stratum.paths import drive_root, repo_root

root = drive_root() / "raw/meteosat"
ds = meteosat.open_day(root, date(2026, 1, 15))
grid = xr.open_zarr(root / "static_grid.zarr")
print(ds)

In [ ]:
# Sanity plot: VZA over India with the 40/50/60° contours and cities; T10.8 for comparison.
cities = {"Mumbai": (19.08, 72.88), "Bangalore": (12.97, 77.59), "Delhi": (28.61, 77.21),
          "Kolkata": (22.57, 88.36), "Guwahati": (26.14, 91.74)}
lat, lon, vza = grid["lat"].values, grid["lon"].values, grid["vza_deg"].values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
m = ax1.scatter(lon[::4, ::4], lat[::4, ::4], c=vza[::4, ::4], s=1, cmap="cividis", vmin=15, vmax=80)
cs = ax1.tricontour(lon[::8, ::8].ravel()[np.isfinite(vza[::8, ::8].ravel())],
                    lat[::8, ::8].ravel()[np.isfinite(vza[::8, ::8].ravel())],
                    vza[::8, ::8].ravel()[np.isfinite(vza[::8, ::8].ravel())], levels=[40, 50, 60], colors="white")
ax1.clabel(cs, fmt="%d°")
for name, (la, lo) in cities.items():
    v = float(meteosat.viewing_zenith_angle(la, lo))
    ax1.plot(lo, la, "o", color="#c0392b", ms=4); ax1.annotate(f"{name} {v:.0f}°", (lo, la), xytext=(4, 4), textcoords="offset points", fontsize=8, color="white")
fig.colorbar(m, ax=ax1, label="viewing zenith angle (°)")
ax1.set_xlim(66, 100); ax1.set_ylim(4, 39); ax1.set_title("Meteosat-9 (45.5°E) viewing angle; VZA ≥ 60° is unobservable", fontsize=10)
t = ds["IR_108"].isel(time=0).values
ax2.scatter(lon[::4, ::4], lat[::4, ::4], c=t[::4, ::4], s=1, cmap="gray_r")
ax2.set_xlim(66, 100); ax2.set_ylim(4, 39); ax2.set_title(f"T10.8 {str(ds.time.values[0])[:16]} UTC (lat/lon view)", fontsize=10)
out = repo_root() / "reports/phase1/meteosat_vza_2026-01-15.png"
fig.tight_layout(); fig.savefig(out, dpi=130)
print(f"share of study box with VZA ≥ 60°: {np.nanmean(vza[(lat >= 6) & (lat <= 37) & (lon >= 68) & (lon <= 98)] >= 60):.1%} | saved {out}")